In [12]:
import vlrdevapi as vlr
import pandas as pd
import re

events = vlr.event.list(tier='vct', return_all=True)

In [ ]:
# filter list of events down to only include international events, get their ID, and then add the actual event object to a list
intl_events = []
for event in events.events:
    if re.search(r"(Masters \w+|Champions 202\d|LOCK//IN)", event.name):
        intl_events.append(vlr.event(event.id))

intl_events

In [ ]:
# create a list of all international matches
intl_matches = []
for event in intl_events:
    match_list = event.matches().matches
    for match in match_list:
        intl_matches.append(vlr.series(match.match_id))

TEST_intl_matches = [intl_matches[0]]
intl_matches

In [45]:
data_dict = {
    "columns": [
        "event_id", "event_name", "series_id", "map_id", "map_name",
        "team_name", "player_name", "player_id", "agent", 
        "acs", "kills", "deaths", "assists", "fkills", "fdeaths"
    ],
    "data": []
}

In [47]:
from vlrdevapi.exceptions import RateLimitError, RequestError

try:
    for seriesObj in intl_matches:
        series = seriesObj.info()
        eventID = series.event_id
        eventName = series.event_name
        seriesID = series.series_id
        for map in series.games:
            mapID = map.game_id
            stats = vlr.series.players(seriesID, mapID)
            mapName = stats.map_name
            for team in [stats.team1, stats.team2]:
                teamName = team.team_name
                for player in team.players:
                    playerID = player.player_id
                    playerName = player.name
                    playerAgent = player.agents[0] if len(player.agents) > 0 else None
                    playerACS = player.stats.overall.acs
                    playerKills = player.stats.overall.kills
                    playerDeaths = player.stats.overall.deaths
                    playerAssists = player.stats.overall.assists
                    playerFKs = player.stats.overall.first_kills
                    playerFDs = player.stats.overall.first_deaths
                    playerRow = [
                        eventID, eventName, seriesID, mapID, mapName,
                        teamName, playerName, playerID, playerAgent,
                        playerACS, playerKills, playerDeaths, playerAssists, playerFKs, playerFDs
                    ]
                    data_dict["data"].append(playerRow)
                    if playerAgent is None: 
                        print(f"No agent found for {playerName} in {eventName}: {stats.team1.team_name} vs. {stats.team2.team_name} (IDs: {seriesID}, {mapID})")
except RequestError:
    print("Error while requesting.")
except RateLimitError:
    print("Too many requests!")

No agent found for TenZ in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for KOHAL in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for Doenmo in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for johnqt in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for karsaj in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for SuperBusS in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for MintAuka in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for cgrs in Valorant Masters Bangkok 2025: Team International vs. Team Thailand (IDs: 450589, 203837)
No agent found for primmie in Valorant Masters B

In [50]:
df = pd.DataFrame(**data_dict)
df.to_csv('data/agent_data.csv')

df

,event_id,event_name,series_id,map_id,map_name,team_name,player_name,player_id,agent,acs,kills,deaths,assists,fkills,fdeaths
0,2766,Valorant Champions 2026,753455,283184,Ascent,Team Liquid,trexx,2168,Sova,241.0,12.0,15.0,5.0,1.0,2.0
1,2766,Valorant Champions 2026,753455,283184,Ascent,Team Liquid,nAts,457,Cypher,188.0,11.0,14.0,2.0,0.0,3.0
2,2766,Valorant Champions 2026,753455,283184,Ascent,Team Liquid,Kicks,11134,Omen,166.0,9.0,14.0,6.0,1.0,1.0
3,2766,Valorant Champions 2026,753455,283184,Ascent,Team Liquid,GSR,50813,Jett,162.0,9.0,15.0,1.0,1.0,2.0
4,2766,Valorant Champions 2026,753455,283184,Ascent,Team Liquid,kamo,14681,Phoenix,195.0,9.0,16.0,4.0,0.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12155,353,Valorant Champions Tour Stage 2: Masters Reykj...,19089,34301,Haven,FNATIC,MAGNUM,2173,Killjoy,261.0,24.0,15.0,4.0,3.0,2.0
12156,353,Valorant Champions Tour Stage 2: Masters Reykj...,19089,34301,Haven,FNATIC,Mistic,1457,Skye,252.0,20.0,20.0,7.0,1.0,1.0
12157,353,Valorant Champions Tour Stage 2: Masters Reykj...,19089,34301,Haven,FNATIC,doma,1369,Sage,200.0,16.0,17.0,10.0,0.0,3.0
12158,353,Valorant Champions Tour Stage 2: Masters Reykj...,19089,34301,Haven,FNATIC,Derke,5022,Jett,223.0,18.0,21.0,4.0,5.0,7.0
